In [ ]:
# -----------------------------------------
# Projection functions
# -----------------------------------------

def project_to_sagittal(vector, fwd, up):
    """
    Project 3D vector into the sagittal plane.
    Returns [forward, up] coordinates.
    """
    return np.stack([
        np.sum(vector * fwd, axis=1),
        np.sum(vector * up, axis=1)
    ], axis=1)

def project_to_frontal(vector, left, up):
    """
    Project 3D vector into the frontal plane.
    Returns [left, up] coordinates.
    """
    return np.stack([
        np.sum(vector * left, axis=1),
        np.sum(vector * up, axis=1)
    ], axis=1)

# -----------------------------------------
# Angle functions
# -----------------------------------------

def angle_between_vectors(v1, v2):
    
    dot = np.sum(v1 * v2, axis=1)

    norm_product = (
        np.linalg.norm(v1, axis=1) *
        np.linalg.norm(v2, axis=1)
    )

    cosine = dot / np.maximum(norm_product, 1e-8)

    return np.degrees(
        np.arccos(
            np.clip(cosine, -1, 1)
        )
    )


### 3D joint angles from keypoints

In [ ]:
# 3D joint angles and plane-specific angles from the extracted keypoints.

required_keypoints = {
    "l_sh_3d",
    "r_sh_3d",
    "l_hip_3d",
    "r_hip_3d",
    "r_knee_3d",
    "r_ankle_3d",
    "r_big_toe_3d",
    "r_small_toe_3d",
}
if not data_log:
    raise ValueError("data_log is empty. Run the SAM 3D extraction cell first.")

missing_keypoints = required_keypoints.difference(data_log[0])
if missing_keypoints:
    missing_names = ", ".join(sorted(missing_keypoints))
    raise ValueError(
        f"data_log is missing {missing_names}. Rerun the extraction cell "
        "after adding the foot keypoints."
    )

sam_df = pd.DataFrame(data_log).sort_values("ts").reset_index(drop=True)

def stack_keypoints(column):
    return np.stack(sam_df[column].values)

l_sh = stack_keypoints("l_sh_3d")
r_sh = stack_keypoints("r_sh_3d")
l_hip = stack_keypoints("l_hip_3d")
r_hip = stack_keypoints("r_hip_3d")
r_knee = stack_keypoints("r_knee_3d")
r_ankle = stack_keypoints("r_ankle_3d")
r_toe = (
    stack_keypoints("r_big_toe_3d")
    + stack_keypoints("r_small_toe_3d")
) / 2


def angle_between_rows(vector_a, vector_b):
    dot = np.sum(vector_a * vector_b, axis=1)
    norm_product = (
        np.linalg.norm(vector_a, axis=1)
        * np.linalg.norm(vector_b, axis=1)
    )
    cosine = dot / np.maximum(norm_product, 1e-8)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


# Spatial included angles use the original 3D coordinates.
mid_hip = (l_hip + r_hip) / 2
mid_sh = (l_sh + r_sh) / 2
trunk_at_hip = mid_sh - r_hip
thigh_at_hip = r_knee - r_hip
thigh_at_knee = r_hip - r_knee
shank_at_knee = r_ankle - r_knee
shank_at_ankle = r_knee - r_ankle
foot_at_ankle = r_toe - r_ankle

hip_included_angle_3d = angle_between_rows(trunk_at_hip, thigh_at_hip)
hip_deviation_from_extension_3d = 180.0 - hip_included_angle_3d

knee_included_angle_3d = angle_between_rows(thigh_at_knee, shank_at_knee)
knee_flexion_magnitude_3d = 180.0 - knee_included_angle_3d

ankle_included_angle_3d = angle_between_rows(shank_at_ankle, foot_at_ankle)


# Build a per-frame anatomical basis for the sagittal/frontal projections.
left = l_hip - r_hip
left_norm = np.linalg.norm(left, axis=1, keepdims=True)
left = left / np.maximum(left_norm, 1e-8)

up = mid_sh - mid_hip
up_norm = np.linalg.norm(up, axis=1, keepdims=True)
up = up / np.maximum(up_norm, 1e-8)

fwd = np.cross(left, up)
fwd_norm = np.linalg.norm(fwd, axis=1, keepdims=True)
fwd = fwd / np.maximum(fwd_norm, 1e-8)

up = np.cross(fwd, left)
up_norm = np.linalg.norm(up, axis=1, keepdims=True)
up = up / np.maximum(up_norm, 1e-8)


def project_sagittal(vector):
    return np.stack((np.sum(vector * fwd, axis=1), np.sum(vector * up, axis=1)), axis=1)


def project_frontal(vector):
    return np.stack((np.sum(vector * left, axis=1), np.sum(vector * up, axis=1)), axis=1)


trunk_sag = project_sagittal(trunk_at_hip)
thigh_hip_sag = project_sagittal(thigh_at_hip)
thigh_knee_sag = project_sagittal(thigh_at_knee)
shank_knee_sag = project_sagittal(shank_at_knee)
shank_ankle_sag = project_sagittal(shank_at_ankle)
foot_ankle_sag = project_sagittal(foot_at_ankle)

trunk_front = project_frontal(trunk_at_hip)
thigh_hip_front = project_frontal(thigh_at_hip)
thigh_knee_front = project_frontal(thigh_at_knee)
shank_knee_front = project_frontal(shank_at_knee)
shank_ankle_front = project_frontal(shank_at_ankle)
foot_ankle_front = project_frontal(foot_at_ankle)

# These are unsigned projected angles; the knee values use 180 degrees as extension.
hip_angle_sag = 180.0 - angle_between_rows(trunk_sag, thigh_hip_sag)
hip_angle_frontal = 180.0 - angle_between_rows(trunk_front, thigh_hip_front)
knee_angle_sag = 180.0 - angle_between_rows(thigh_knee_sag, shank_knee_sag)
knee_angle_frontal = 180.0 - angle_between_rows(thigh_knee_front, shank_knee_front)
ankle_angle_sag = angle_between_rows(shank_ankle_sag, foot_ankle_sag)
ankle_angle_frontal = angle_between_rows(shank_ankle_front, foot_ankle_front)

In [ ]:
from scipy.signal import savgol_filter

def smooth_geometric_angle(angle_deg, window=9, poly=3):
    """
    Applies Savitzky-Golay filtering directly to a 1D scalar geometric angle.
    Ensures window is odd and within array bounds.
    """
    angle_arr = np.asarray(angle_deg, dtype=float)
    n_samples = len(angle_arr)
    
    # Window length must be odd and <= data length
    if window >= n_samples:
        window = n_samples - 1 if (n_samples % 2 == 0) else n_samples
    if window < poly + 2:
        return angle_arr  # Return raw if signal is too short to filter
        
    return savgol_filter(angle_arr, window_length=window, polyorder=poly)

# --- Apply to your geometric angles ---
hip_sag_smooth = smooth_geometric_angle(hip_angle_sag, window=9, poly=3)
knee_sag_smooth = smooth_geometric_angle(knee_angle_sag, window=9, poly=3)
ankle_sag_smooth = smooth_geometric_angle(ankle_angle_sag, window=9, poly=3)

hip_front_smooth = smooth_geometric_angle(hip_angle_frontal, window=9, poly=3)
knee_front_smooth = smooth_geometric_angle(knee_angle_frontal, window=9, poly=3)
ankle_front_smooth = smooth_geometric_angle(ankle_angle_frontal, window=9, poly=3)



In [ ]:
# from scipy.signal import savgol_filter # best for smoothing parametric curves offline (kalman for online)
from scipy.spatial.transform import Rotation as R

def smooth_parametric_rotations(matrix_list, window=8, poly=3):
    stacked_matrices = np.stack(matrix_list)
    r_objects = R.from_matrix(stacked_matrices)
    
    # 1. Convert to Quaternions for safe mathematical smoothing
    quats = r_objects.as_quat()
    
    # 2. Enforce Quaternion Continuity (prevent sign flips)
    for i in range(1, len(quats)):
        if np.dot(quats[i], quats[i-1]) < 0:
            quats[i] = -quats[i]
            
    # 3. Apply Savitzky-Golay to the Latent Quaternions
    smoothed_quats = np.zeros_like(quats)
    for i in range(4): 
        smoothed_quats[:, i] = savgol_filter(quats[:, i], window, poly)
        
    # Re-normalize
    smoothed_quats = smoothed_quats / np.linalg.norm(smoothed_quats, axis=1, keepdims=True)

    # 4. Convert to Euler Angles ('xyz' typically maps X to sagittal flexion in SMPL/MHR but I need to VERIFY this!)
    raw_euler = r_objects.as_euler('xyz', degrees=True)
    smoothed_euler = R.from_quat(smoothed_quats).as_euler('xyz', degrees=True)
    
    return raw_euler, smoothed_euler

if not data_log:
    print("❌ ERROR: data_log is empty!")
else:
    df = pd.DataFrame(data_log).sort_values("ts").reset_index(drop=True)
    
    # Process Hip, Knee and Ankle Rotations
    hip_raw, hip_smooth = smooth_parametric_rotations(df["R_rhip_local"].tolist())
    knee_raw, knee_smooth = smooth_parametric_rotations(df["R_rknee_local"].tolist())
    ankle_raw, ankle_smooth = smooth_parametric_rotations(df["R_rankle_local"].tolist())

    
    # Extract the Sagittal Flexion Axis
    # Depending on the MHR resting pose, Sagittal Flexion is usually the X axis (index 0) or Z axis (index 2)
    SAGITTAL_AXIS = 0 
    
    df["sam_r_hip_flex"] = hip_smooth[:, SAGITTAL_AXIS]
    df["sam_r_knee_flex"] = knee_smooth[:, SAGITTAL_AXIS]
    df["sam_r_ankle_flex"] = ankle_smooth[:, SAGITTAL_AXIS]

    print("Parametric Smoothing Complete!")